# TrialMatch — Fine-tuning Gemma 4 with Unsloth

**Before running this notebook:**
1. In the top menu: `Runtime → Change runtime type → T4 GPU` (free tier)
2. Upload `training_data.jsonl` to your Google Drive
3. Fill in your Hugging Face token and username in Step 4

Then run each cell top to bottom. The whole thing takes 1–3 hours.

## Step 1 — Install Unsloth and dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade trl transformers accelerate peft datasets

## Step 2 — Load Gemma 4 with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# This downloads and loads the same Gemma 4 model used in TrialMatch,
# optimised by Unsloth for fast fine-tuning on a free GPU.
# If this model name gives a 404, go to https://huggingface.co/unsloth
# and search for 'gemma-4' to find the exact current name.
MODEL_NAME     = "unsloth/gemma-3-4b-it-bnb-4bit"  # update if needed
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = True,
)
print(f"Loaded: {MODEL_NAME}")

## Step 3 — Add LoRA adapters (what actually gets fine-tuned)

In [ ]:
# LoRA = Low-Rank Adaptation. Instead of retraining the whole model
# (which would need 100s of GBs of memory), we add small trainable
# layers on top. Much faster, much cheaper, almost as good.
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    target_modules     = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
    lora_alpha         = 16,
    lora_dropout       = 0,
    bias               = "none",
    use_gradient_checkpointing = "unsloth",
    random_state       = 42,
)
print("LoRA adapters added.")

## Step 4 — Mount Google Drive and load training data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path if you saved training_data.jsonl somewhere else in Drive
TRAINING_DATA_PATH = "/content/drive/MyDrive/training_data.jsonl"

import json
examples = []
with open(TRAINING_DATA_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            examples.append(json.loads(line))

print(f"Loaded {len(examples)} training examples.")
match_n   = sum(1 for e in examples if e['verdict'] == 'MATCH')
partial_n = sum(1 for e in examples if e['verdict'] == 'PARTIAL')
no_n      = sum(1 for e in examples if e['verdict'] == 'NO')
print(f"  MATCH: {match_n}  PARTIAL: {partial_n}  NO: {no_n}")

## Step 5 — Format data into Gemma chat format

In [ ]:
from datasets import Dataset

# This is the exact same prompt the app uses at runtime.
# We teach the model to produce perfectly structured output every time.
PROMPT_TEMPLATE = """<start_of_turn>user
You are a clinical trial eligibility screener.

PATIENT PROFILE:
{patient_profile}

CLINICAL REASONING SUMMARY:
{clinical_reasoning}

TRIAL ELIGIBILITY CRITERIA:
{eligibility_criteria}

Based solely on the information provided, determine if this patient qualifies.

Output ONLY the following five lines:
VERDICT: [MATCH / PARTIAL / NO]
CONFIDENCE: [0-100]
REASON: [one plain-English sentence]
DISQUALIFIERS: [exact reason or NONE]
NEXT STEP: [one sentence for the patient]<end_of_turn>
<start_of_turn>model
VERDICT: {verdict}
CONFIDENCE: {confidence}
REASON: {reason}
DISQUALIFIERS: {disqualifiers}
NEXT STEP: {next_step}<end_of_turn>"""

def format_example(ex):
    return {
        "text": PROMPT_TEMPLATE.format(
            patient_profile      = ex["patient_profile"][:700],
            clinical_reasoning   = ex["clinical_reasoning"][:350],
            eligibility_criteria = ex["eligibility_criteria"][:1500],
            verdict              = ex["verdict"],
            confidence           = ex["confidence"],
            reason               = ex["reason"],
            disqualifiers        = ex["disqualifiers"],
            next_step            = ex["next_step"],
        )
    }

dataset = Dataset.from_list([format_example(e) for e in examples])
print(f"Dataset ready: {len(dataset)} examples")
print("\nSample (first 300 chars):")
print(dataset[0]['text'][:300])

## Step 6 — Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = dataset,
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,
        warmup_steps                 = 5,
        num_train_epochs             = 3,
        learning_rate                = 2e-4,
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        logging_steps                = 10,
        optim                        = "adamw_8bit",
        weight_decay                 = 0.01,
        lr_scheduler_type            = "linear",
        seed                         = 42,
        output_dir                   = "/content/trialmatch_outputs",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nDone. Training took {trainer_stats.metrics['train_runtime']:.0f} seconds.")

## Step 7 — Upload to Hugging Face

1. Go to **huggingface.co → Settings → Access Tokens → New Token** (write access)
2. Paste the token below where it says `YOUR_HF_TOKEN`
3. Replace `YOUR_HF_USERNAME` with your Hugging Face username

In [ ]:
HF_TOKEN    = "YOUR_HF_TOKEN"      # paste your Hugging Face token here
HF_USERNAME = "YOUR_HF_USERNAME"   # your Hugging Face username
REPO_NAME   = f"{HF_USERNAME}/trialmatch-gemma4"

print(f"Uploading to: https://huggingface.co/{REPO_NAME}")
print("This may take 5–15 minutes depending on model size...")

model.push_to_hub_merged(
    REPO_NAME,
    tokenizer,
    save_method = "lora",   # uploads only the small LoRA adapters (~50MB)
    token       = HF_TOKEN,
)

print(f"\nDone! Your model is live at:")
print(f"https://huggingface.co/{REPO_NAME}")
print("\nCopy that URL — you'll need it for your Kaggle writeup and demo video.")

## All done! Next step: run benchmark_results.py on your local machine.